# TF-IDF Baseline v1

한능검 ML v1 데이터로 첫 baseline 성능을 확인하는 노트북입니다.

이번 노트북은 `split_v1` 구조를 사용합니다.

진행 흐름:
1. Google Drive 연결
2. 입력 파일 확인
3. train / predict_input / test_answer 데이터 로드
4. 라벨 분포와 라벨 제거 여부 확인
5. TF-IDF baseline 함수 정의
6. `era`, `topic`, `question_type` 순서로 학습/예측/평가
7. 결과 파일 저장

핵심 구조:

```text
train_features_v1      -> 학습용, 정답 라벨 있음
predict_input_v1       -> 예측용, era/topic/question_type 빈칸
test_answer_v1         -> 채점용, 정답 라벨 있음
```

이 baseline은 딥러닝이 아니므로 CPU 런타임으로 충분합니다.

## 1. Google Drive 연결

`Final_project` 폴더가 있는 Google Drive를 Colab에 연결합니다.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2. 경로 설정 및 파일 확인

`common/split_v1` 폴더 안의 세 파일과 `ml_han_class_weights_v1.json`이 모두 `exists = True`로 나와야 합니다.

In [ ]:
from pathlib import Path

BASE_DIR = Path('/content/drive/MyDrive/Final_project')
COMMON_DIR = BASE_DIR / 'common'
SPLIT_DIR = COMMON_DIR / 'split_v1'
RESULT_DIR = COMMON_DIR / 'baseline_tfidf_v1'

TRAIN_JSON = SPLIT_DIR / 'train_features_v1.json'
PREDICT_JSON = SPLIT_DIR / 'predict_input_v1.json'
ANSWER_JSON = SPLIT_DIR / 'test_answer_v1.json'
CLASS_WEIGHT_JSON = COMMON_DIR / 'ml_han_class_weights_v1.json'

RESULT_JSON = RESULT_DIR / 'baseline_tfidf_results_v1.json'
RESULT_MD = RESULT_DIR / 'baseline_tfidf_results_v1.md'

TARGET_COLUMNS = ['era', 'topic', 'question_type']

print('BASE_DIR:', BASE_DIR)
print('COMMON_DIR:', COMMON_DIR)
print('SPLIT_DIR:', SPLIT_DIR)
print()
for path in [TRAIN_JSON, PREDICT_JSON, ANSWER_JSON, CLASS_WEIGHT_JSON]:
    print(path.name, 'exists =', path.exists())

## 3. 라이브러리 불러오기

Colab에는 보통 `scikit-learn`이 기본 설치되어 있습니다. import 오류가 나면 주석 처리된 설치 명령을 실행하세요.

In [ ]:
# import 오류가 날 때만 아래 주석을 풀고 실행하세요.
# !pip install -q scikit-learn

import json
import csv
from collections import Counter
from typing import Any

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, f1_score
from sklearn.pipeline import Pipeline

print('libraries loaded')

## 4. 데이터 로드 함수 정의

JSON 파일을 읽는 함수입니다. 이 단계에서는 아직 학습을 하지 않습니다.

In [ ]:
def read_json(path: Path):
    return json.loads(path.read_text(encoding='utf-8'))

## 5. 데이터 로드

정상이라면 다음 행 수가 나와야 합니다.

- train: 1200
- predict input: 400
- test answer: 400

In [ ]:
train_rows = read_json(TRAIN_JSON)
predict_rows = read_json(PREDICT_JSON)
answer_rows = read_json(ANSWER_JSON)
assets = read_json(CLASS_WEIGHT_JSON)

print('train rows:', len(train_rows))
print('predict rows:', len(predict_rows))
print('answer rows:', len(answer_rows))
print('targets:', TARGET_COLUMNS)
print('asset targets:', assets['target_columns'])

## 6. 샘플 데이터 확인

`train_features_v1`에는 정답 라벨이 있고, `predict_input_v1`에는 예측해야 할 라벨이 빈칸이어야 합니다.

In [ ]:
train_sample = train_rows[0]
predict_sample = predict_rows[0]
answer_sample = answer_rows[0]

print('[train sample labels]')
print({target: train_sample.get(target) for target in TARGET_COLUMNS})
print('\n[predict sample labels: should be blank]')
print({target: predict_sample.get(target) for target in TARGET_COLUMNS})
print('\n[answer sample labels]')
print({target: answer_sample.get(target) for target in TARGET_COLUMNS})
print('\n[predict text preview]')
print(predict_sample['text'][:500])

## 7. Train/Test 정답 라벨 분포 확인

학습용 train 라벨 분포와 채점용 answer 라벨 분포를 확인합니다. `predict_input_v1`의 예측 대상 라벨은 빈칸이어야 합니다.

In [ ]:
def label_counts(rows: list[dict], target: str):
    return Counter(str(row.get(target) or '') for row in rows)

print('[predict_input label blank check]')
for target in TARGET_COLUMNS:
    blanks = sum(1 for row in predict_rows if not row.get(target))
    print(target, 'blank rows =', blanks)

for target in TARGET_COLUMNS:
    print('\n===', target, '===')
    print('[train]')
    for label, count in label_counts(train_rows, target).most_common():
        print(label, count)
    print('[test answer]')
    for label, count in label_counts(answer_rows, target).most_common():
        print(label, count)

## 8. Class Weight 확인

소수 라벨일수록 weight가 크게 나와야 합니다. 이 weight는 Logistic Regression 학습에 사용됩니다.

In [ ]:
for target in TARGET_COLUMNS:
    print('\n===', target, 'class weights ===')
    weights = assets['assets'][target]['class_weights']
    for label, weight in sorted(weights.items(), key=lambda x: x[1], reverse=True):
        print(label, weight)

## 9. 모델 입력/라벨 추출 함수

학습에는 `train_features_v1`의 `text`와 정답 라벨을 사용합니다.
예측에는 `predict_input_v1`의 `text`만 사용합니다.
평가에는 `test_answer_v1`의 정답 라벨을 사용합니다.

In [ ]:
def get_texts(rows: list[dict]) -> list[str]:
    return [str(row.get('text') or '') for row in rows]


def get_labels(rows: list[dict], target: str) -> list[str]:
    return [str(row.get(target) or '') for row in rows]


def get_class_weight(assets: dict, target: str) -> dict[str, float]:
    return {
        label: float(weight)
        for label, weight in assets['assets'][target]['class_weights'].items()
    }

## 10. TF-IDF + Logistic Regression 모델 함수

- TF-IDF: 텍스트를 글자 n-gram 중요도 벡터로 변환합니다.
- Logistic Regression: 변환된 벡터로 라벨을 분류합니다.
- `class_weight`: 다수 라벨 쏠림을 줄이고 소수 라벨을 더 크게 반영합니다.

In [ ]:
def build_pipeline(class_weight: dict[str, float]):
    return Pipeline(
        steps=[
            (
                'tfidf',
                TfidfVectorizer(
                    analyzer='char_wb',
                    ngram_range=(2, 5),
                    min_df=2,
                    max_features=80000,
                    sublinear_tf=True,
                ),
            ),
            (
                'clf',
                LogisticRegression(
                    max_iter=2000,
                    class_weight=class_weight,
                    solver='liblinear',
                    random_state=42,
                ),
            ),
        ]
    )

## 11. 평가 함수 정의

Accuracy는 참고용이고, 라벨 인밸런스가 있으므로 `Macro F1`을 중요하게 봅니다.

이 함수는 다음 순서로 동작합니다.

```text
train_features_v1로 학습
predict_input_v1로 예측
test_answer_v1과 비교해서 평가
```

In [ ]:
def evaluate_predictions(y_true: list[str], y_pred: list[str]) -> dict:
    return {
        'accuracy': round(float(accuracy_score(y_true, y_pred)), 6),
        'macro_f1': round(float(f1_score(y_true, y_pred, average='macro', zero_division=0)), 6),
        'weighted_f1': round(float(f1_score(y_true, y_pred, average='weighted', zero_division=0)), 6),
        'classification_report': classification_report(
            y_true,
            y_pred,
            output_dict=True,
            zero_division=0,
        ),
    }


def train_one_target(train_rows: list[dict], predict_rows: list[dict], answer_rows: list[dict], assets: dict, target: str) -> dict:
    x_train = get_texts(train_rows)
    y_train = get_labels(train_rows, target)
    x_predict = get_texts(predict_rows)
    y_true = get_labels(answer_rows, target)

    model = build_pipeline(get_class_weight(assets, target))
    model.fit(x_train, y_train)
    y_pred = model.predict(x_predict).tolist()

    row_predictions = []
    for predict_row, answer_row, true_label, pred_label in zip(predict_rows, answer_rows, y_true, y_pred):
        row_predictions.append(
            {
                'round_no': predict_row.get('round_no'),
                'question_no': predict_row.get('question_no'),
                'problem_id': predict_row.get('problem_id'),
                'true_label': true_label,
                'pred_label': pred_label,
                'is_correct': true_label == pred_label,
                'text_preview': str(predict_row.get('text') or '')[:160].replace('\n', ' '),
            }
        )

    return {
        'target': target,
        'train_counts': dict(Counter(y_train).most_common()),
        'test_counts': dict(Counter(y_true).most_common()),
        'pred_counts': dict(Counter(y_pred).most_common()),
        'metrics': evaluate_predictions(y_true, y_pred),
        'row_predictions': row_predictions,
    }

## 12. era 모델 학습/평가

먼저 시대 분류 모델만 실행합니다. 결과가 나오면 `accuracy`, `macro_f1`, `weighted_f1`을 확인하세요.

In [ ]:
results = {
    'base_dir': str(BASE_DIR),
    'common_dir': str(COMMON_DIR),
    'split_dir': str(SPLIT_DIR),
    'train_rows': len(train_rows),
    'predict_rows': len(predict_rows),
    'answer_rows': len(answer_rows),
    'targets': {},
}

results['targets']['era'] = train_one_target(train_rows, predict_rows, answer_rows, assets, 'era')
results['targets']['era']['metrics']['accuracy'], results['targets']['era']['metrics']['macro_f1'], results['targets']['era']['metrics']['weighted_f1']

## 13. topic 모델 학습/평가

주제 분류 모델을 실행합니다.

In [ ]:
results['targets']['topic'] = train_one_target(train_rows, predict_rows, answer_rows, assets, 'topic')
results['targets']['topic']['metrics']['accuracy'], results['targets']['topic']['metrics']['macro_f1'], results['targets']['topic']['metrics']['weighted_f1']

## 14. question_type 모델 학습/평가

문항 유형은 인밸런스가 가장 큰 타깃입니다. Accuracy보다 Macro F1과 라벨별 성능을 더 중요하게 확인해야 합니다.

In [ ]:
results['targets']['question_type'] = train_one_target(train_rows, predict_rows, answer_rows, assets, 'question_type')
results['targets']['question_type']['metrics']['accuracy'], results['targets']['question_type']['metrics']['macro_f1'], results['targets']['question_type']['metrics']['weighted_f1']

## 15. 전체 요약 확인

세 모델의 핵심 지표를 한 번에 확인합니다.

In [ ]:
summary = {
    target: {
        key: results['targets'][target]['metrics'][key]
        for key in ['accuracy', 'macro_f1', 'weighted_f1']
    }
    for target in TARGET_COLUMNS
}
summary

## 16. Markdown 리포트 생성 함수

결과를 파일로 저장하기 전에 사람이 읽기 좋은 Markdown 리포트로 변환합니다.

In [ ]:
def build_markdown(results: dict) -> str:
    lines = []
    lines.append('# TF-IDF Baseline Results v1')
    lines.append('')
    lines.append('- Train data: `split_v1/train_features_v1.json`')
    lines.append('- Predict input: `split_v1/predict_input_v1.json`')
    lines.append('- Test answer: `split_v1/test_answer_v1.json`')
    lines.append('- Model: `TfidfVectorizer(char_wb 2~5gram) + LogisticRegression`')
    lines.append('- Imbalance handling: train-based `class_weight`')
    lines.append('- Split: train 47~70, test 71~78')
    lines.append('')
    lines.append('## Summary')
    lines.append('')
    lines.append('| target | accuracy | macro_f1 | weighted_f1 |')
    lines.append('|---|---:|---:|---:|')

    for target in TARGET_COLUMNS:
        metrics = results['targets'][target]['metrics']
        lines.append(
            f"| {target} | {metrics['accuracy']:.4f} | "
            f"{metrics['macro_f1']:.4f} | {metrics['weighted_f1']:.4f} |"
        )
    lines.append('')

    for target in TARGET_COLUMNS:
        target_result = results['targets'][target]
        report = target_result['metrics']['classification_report']
        labels = sorted(
            set(target_result['train_counts'])
            | set(target_result['test_counts'])
            | set(target_result['pred_counts'])
        )

        lines.append(f'## {target}')
        lines.append('')
        lines.append('### Label Distribution')
        lines.append('')
        lines.append('| label | train | test | pred |')
        lines.append('|---|---:|---:|---:|')
        for label in labels:
            lines.append(
                f"| {label} | {target_result['train_counts'].get(label, 0)} | "
                f"{target_result['test_counts'].get(label, 0)} | "
                f"{target_result['pred_counts'].get(label, 0)} |"
            )
        lines.append('')
        lines.append('### Per-class Metrics')
        lines.append('')
        lines.append('| label | precision | recall | f1-score | support |')
        lines.append('|---|---:|---:|---:|---:|')
        for label in labels:
            values = report.get(label, {})
            lines.append(
                f"| {label} | {values.get('precision', 0):.4f} | "
                f"{values.get('recall', 0):.4f} | "
                f"{values.get('f1-score', 0):.4f} | "
                f"{int(values.get('support', 0))} |"
            )
        lines.append('')
    return '\n'.join(lines) + '\n'

## 17. 결과 저장

JSON과 Markdown 결과를 `common/baseline_tfidf_v1` 폴더에 저장합니다.

In [ ]:
RESULT_DIR.mkdir(parents=True, exist_ok=True)
RESULT_JSON.write_text(json.dumps(results, ensure_ascii=False, indent=2) + '\n', encoding='utf-8')
RESULT_MD.write_text(build_markdown(results), encoding='utf-8')

for target in TARGET_COLUMNS:
    pred_csv = RESULT_DIR / f'{target}_predictions_v1.csv'
    with pred_csv.open('w', encoding='utf-8-sig', newline='') as file:
        import csv
        writer = csv.DictWriter(
            file,
            fieldnames=[
                'round_no',
                'question_no',
                'problem_id',
                'true_label',
                'pred_label',
                'is_correct',
                'text_preview',
            ],
        )
        writer.writeheader()
        writer.writerows(results['targets'][target]['row_predictions'])
    print('saved predictions:', pred_csv)

print('saved json:', RESULT_JSON)
print('saved md:', RESULT_MD)

## 18. 저장된 Markdown 결과 확인

저장된 리포트 앞부분을 출력합니다. 이 파일을 내려받아 공유하면 다음 평가 문서에 반영할 수 있습니다.

In [ ]:
print(RESULT_MD.read_text(encoding='utf-8')[:4000])

## 19. 행별 예측 결과 확인

`true_label`은 숨겨둔 정답이고, `pred_label`은 모델이 `predict_input_v1`의 text만 보고 예측한 값입니다.

In [ ]:
# Preview row-level prediction results.
# true_label comes from test_answer_v1.
# pred_label is predicted from predict_input_v1 text only.
import csv

preview_path = RESULT_DIR / 'era_predictions_v1.csv'
with preview_path.open('r', encoding='utf-8-sig') as file:
    reader = csv.DictReader(file)
    for index, row in enumerate(reader):
        print(row)
        if index >= 4:
            break
